<a href="https://colab.research.google.com/github/Flamers-Team/Techchalleng3/blob/main/notebooks/conexao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:

# 1. Resetar pacotes conflitantes
!pip uninstall -y gradio pydantic fastapi starlette 2>&1 | tail -3

# 2. Instalar chromadb PRIMEIRO (versão antiga que funciona)
!pip install -q "numpy<2.0"
!pip install -q "chromadb==0.4.18" "pydantic==1.10.13"

# 3. Instalar gradio 4.x (compatível com pydantic 1.x)
!pip install -q "gradio==4.44.0"

# 4. Instalar o resto sem mexer em pydantic/numpy
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-deps
!pip install -q transformers==4.46.3 peft==0.13.0 bitsandbytes==0.45.0 accelerate==1.1.1 trl==0.12.0
!pip install -q sentence-transformers

# 5. Verificar
import pydantic, gradio, chromadb, numpy
print(f"✅ pydantic={pydantic.__version__}")
print(f"✅ gradio={gradio.__version__}")
print(f"✅ chromadb={chromadb.__version__}")
print(f"✅ numpy={numpy.__version__}")

Found existing installation: starlette 0.50.0
Uninstalling starlette-0.50.0:
  Successfully uninstalled starlette-0.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.9.2 requires bitsandbytes!=0.46.0,!=0.48.0,>=0.45.5, but you have bitsandbytes 0.45.0 which is incompatible.
unsloth 2026.9.2 requires peft!=0.11.0,>=0.18.0, but you have peft 0.13.0 which is incompatible.
unsloth 2026.9.2 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 4.46.3 which is incompatible.
unsloth 2026.9.2 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 0.12.0 which is incompatible.
gradio-client 1.3.0 requires websockets<13.0,>=10.0, but you have websockets 17.1 which is incompatible.
pydantic-settings 2.15.0 requires pydantic>=2.7.0

In [2]:
import os, shutil
from pathlib import Path

# 1. Ir pro repo (já clonado antes)
%cd /content/Techchalleng3

# 2. Confirmar que o modelo está no lugar
MODELO = Path("/content/biomistral-medquad-lora")
if not MODELO.exists():
    DRIVE = Path("/content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora")
    if DRIVE.exists():
        shutil.copytree(DRIVE, MODELO)
        print(f"✅ Modelo copiado do Drive")
    else:
        print("❌ Modelo não encontrado. Rode o fine-tuning antes.")
        raise FileNotFoundError("Modelo não existe")

# 3. Verificar GPU
import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# 4. Subir UI
os.chdir("/content/Techchalleng3")
!python src/ui/gradio_app.py

/content/Techchalleng3
✅ GPU: NVIDIA A100-SXM4-40GB
🏥 INICIANDO ASSISTENTE MÉDICO INTELIGENTE (VERSÃO FINAL)
ℹ️  LoRA adapters não encontrados em biomistral-medquad-lora
   Usando modo MOCK até você rodar o fine-tuning
⚠️  Retriever não carregado: ChromaDB não encontrado em /content/Techchalleng3/data/processed/chroma_index. Rode primeiro: python src/rag/build_index_chatbulario.py
✅ Todos os componentes inicializados!
/usr/local/lib/python3.13/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(
Running on local URL:  http://0.0.0.0:7860
ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, s

In [8]:
!pip install -q "gradio==4.16.0" "gradio_client==0.16.0"
!python src/ui/gradio_app.py

ERROR: Cannot install gradio==4.16.0 and gradio_client==0.16.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
Traceback (most recent call last):
  File "/content/Techchalleng3/src/ui/gradio_app.py", line 26, in <module>
    import gradio as gr
  File "/usr/local/lib/python3.13/dist-packages/gradio/__init__.py", line 3, in <module>
    import gradio._simple_templates
  File "/usr/local/lib/python3.13/dist-packages/gradio/_simple_templates/__init__.py", line 1, in <module>
    from .simpledropdown import SimpleDropdown
  File "/usr/local/lib/python3.13/dist-packages/gradio/_simple_templates/simpledropdown.py", line 6, in <module>
    from gradio.components.base import Component, FormComponent
  File "/usr/local/lib/python3.13/dist-packages/gradio/components/__init__.py", line 1, in <module>
    from gradio.components.annotated_image impo

In [9]:
!python src/rag/build_index_chatbulario.py

📥 INDEXAÇÃO DO CHATBULÁRIO NO CHROMADB

📂 Carregando amostras do ChatBulário...
⚠️  Arquivo não encontrado: /content/Techchalleng3/data/raw/chatbulario_train.jsonl
   Total disponível: 11,478 pares Q&A

🔌 Conectando ao ChromaDB em /content/Techchalleng3/data/processed/chroma_index...
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-09-02 15:43:58.033936: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-02 15:43:58.104094: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate c

In [10]:
# ================================================================
# 🎨 UI MINIMALISTA - Funciona com qualquer gradio
# ================================================================

import os, json
from pathlib import Path

os.chdir("/content/Techchalleng3")

# Instala reportlab se faltar
!pip install -q reportlab

# Carrega RAG (ChatBulário)
print("📚 Carregando RAG...")
from src.rag.retriever import Retriever
retriever = Retriever()

# Carrega LLM real (se modelo existe)
print("🤖 Carregando LLM...")
from src.llm.client import LLMClient
MODELO = Path("/content/biomistral-medquad-lora")
llm = LLMClient(lora_path=MODELO if MODELO.exists() else Path("biomistral-medquad-lora"))
print(f"   Modo: {'✅ LLM REAL (fine-tuned)' if not llm.use_mock else '⚠️ MOCK (modelo não encontrado)'}")

# Carrega agentes
from src.agents.triagem import TriagemAgent
from src.agents.sintese import SinteseAgent
from src.agents.validacao import ValidacaoAgent

triagem = TriagemAgent(llm=llm)
sintese = SinteseAgent(llm=llm)
validacao = ValidacaoAgent(llm=llm)

# Função principal
def consulta(relato, nome, idade):
    try:
        print(f"\n{'='*60}\n📋 TRIAGEM\n{'='*60}")
        r1 = triagem.executar(relato)
        print(f"   {r1.get('categoria', '?')}: {r1.get('justificativa', '?')[:150]}")

        print(f"\n📚 RAG...")
        chunks = retriever.retrieve_interno(relato, k=5)
        fontes = [f"[{c['rag_source']}] {c['source']}" for c in chunks]
        contexto = retriever.formatar_contexto(chunks)
        print(f"   {len(chunks)} bulas encontradas")

        print(f"\n🧠 SÍNTESE...")
        r2 = sintese.executar(relato, contexto)
        r3 = validacao.executar(r2)

        saida_t = json.dumps(r1, indent=2, ensure_ascii=False)
        saida_r = "\n".join(fontes) if fontes else "(nenhum resultado)"
        saida_s = json.dumps(r3, indent=2, ensure_ascii=False)
        return saida_t, saida_r, saida_s
    except Exception as e:
        return f"❌ ERRO: {e}", "", ""

# UI Gradio minimalista
import gradio as gr

with gr.Blocks(title="🏥 Assistente Médico") as demo:
    gr.Markdown("# 🏥 Assistente Médico com LLM Fine-Tuned\n**Tech Challenge FIAP - Fase 3**")

    with gr.Row():
        with gr.Column():
            nome = gr.Textbox(label="Nome", value="Maria Silva")
            idade = gr.Textbox(label="Idade", value="45")
            relato = gr.Textbox(
                label="Relato Clínico",
                placeholder="Ex: dor torácica há 3h, irradiando para braço...",
                lines=5,
            )
            btn = gr.Button("🔬 Analisar")

        with gr.Column():
            out_t = gr.Textbox(label="📋 Triagem", lines=8)
            out_r = gr.Textbox(label="📚 Fontes RAG", lines=6)
            out_s = gr.Textbox(label="🧠 Síntese Final", lines=15)

    btn.click(fn=consulta, inputs=[relato, nome, idade], outputs=[out_t, out_r, out_s])

print("\n🚀 Subindo UI...")
demo.launch(share=True, server_name="0.0.0.0", debug=False)

📚 Carregando RAG...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


RuntimeError: Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
module 'numpy.dtypes' has no attribute 'StringDType'